## Field Level Comparison Report - Style
*Anchor: `catalog-stream-dbx-silver` &nbsp;| `catalog-load-dbx-silver` &nbsp; Compare: * 

In [0]:
from sdds.common.util import NotebookUtil
import pandas as pd
import requests
from datetime import datetime

from pyspark.sql.functions import col, lit

# -------------------------------------------------------
# ADD / REMOVE RECIPIENTS HERE
RECIPIENTS = [
    "rafael.miranda@dcsg.com",
    "leonardo.paschoal@dcsg.com",
    # "product-team@dcsg.com",
]
FLOW_URL = "https://defaulte04b15c87a1e43909b5b28c7c205a2.33.environment.api.powerplatform.com:443/powerautomate/automations/direct/workflows/df8de47253794650b48d45d3f26fd871/triggers/manual/paths/invoke?api-version=1"
# -------------------------------------------------------

catalog_name    = NotebookUtil.notebook_param("sdds_catalog")
gold_schema     = NotebookUtil.notebook_param("sdds_gold_schema")
metadata_table  = f"`{catalog_name}`.`{gold_schema}`.`comparison_run_metadata_style`"
summary_table   = f"`{catalog_name}`.`{gold_schema}`.`field_level_comparison_summary_style`"
run_detail_view = f"`{catalog_name}`.`{gold_schema}`.`view_comparison_run_detail_style`"

# pull latest run
run_meta = spark.table(metadata_table)\
    .orderBy(col("run_timestamp").desc())\
    .limit(7)\
    .collect()

latest_run_id    = run_meta[0]["run_id"]
latest_run_date  = str(run_meta[0]["run_date"])
latest_run_date_timestamp = str(datetime.now().strftime("%Y-%m-%d:%H:%M:%S"))
latest_run_date_val = run_meta[0]["run_date"]
anchor_table     = run_meta[0]["anchor_table"]
compare_table    = run_meta[0]["compare_table"]
match_total      = int(run_meta[0]["match_count"])
mismatch_total   = int(run_meta[0]["mismatch_count"])
missing_total    = int(run_meta[0]["missing_count"])
extra_total      = int(run_meta[0]["extra_count"])

# field-level data for current run
df = (
    spark.table(run_detail_view)
    .filter(col("run_id") == latest_run_id)
    .orderBy("field")
    .toPandas()
)

# -------------------------------------------------------
# TREND: compare current similarity_pct vs the most recent PRIOR-DAY run
# (day-over-day, so re-running today does not zero out the trend)
# -------------------------------------------------------
prior_run_row = (
    spark.table(summary_table)
    .select("run_id", "run_date", "run_timestamp")
    .distinct()
    .orderBy(col("run_timestamp").desc())
    .select("run_id", "run_date")
    .limit(7)
    .collect()
)
latest_run_id    = run_meta[0]["run_id"]
latest_run_date  = str(run_meta[0]["run_date"])
latest_run_date_val = run_meta[0]["run_date"]
anchor_table     = run_meta[0]["anchor_table"]
compare_table    = run_meta[0]["compare_table"]
match_total      = int(run_meta[0]["match_count"])
mismatch_total   = int(run_meta[0]["mismatch_count"])
missing_total    = int(run_meta[0]["missing_count"])
extra_total      = int(run_meta[0]["extra_count"])

# field-level data for current run
df = (
    spark.table(run_detail_view)
    .filter(col("run_id") == latest_run_id)
    .orderBy("field")
    .toPandas()
)

# -------------------------------------------------------
# TREND: compare current similarity_pct vs the most recent PRIOR-DAY run
# (day-over-day, so re-running today does not zero out the trend)
# -------------------------------------------------------
prior_run_row = (
    spark.table(summary_table)
    .select("run_id", "run_date", "run_timestamp")
    .distinct()
    .orderBy(col("run_timestamp").desc())
    .select("run_id", "run_date")
    .limit(7)
    .collect()
)
for i in range(0, 6):
    if prior_run_row:
        prior_run_id   = prior_run_row[i]["run_id"]
        prior_run_date = str(prior_run_row[i]["run_date"])
        prior_df = (
            spark.table(summary_table)
            .filter(col("run_id") == prior_run_id)
            .select("field", col("similarity_pct").alias(f"prior_similarity_pct_{i}_days_ago"))
            .toPandas()
        )
        df = df.merge(prior_df, on="field", how="left")
        df[f"trend_{i}_days_ago"] = df["similarity_pct"] - df[f"prior_similarity_pct_{i}_days_ago"]

    else:
        prior_run_date = None
        df[f"prior_similarity_pct_{i}_days_ago"] = pd.NA
        df[f"trend_{i}_days_ago"] = pd.NA

# -------------------------------------------------------
# TREND: compare current similarity_pct vs the last N PRIOR-DAY runs
# (day-over-day, so re-running today does not zero out the trend)
# -------------------------------------------------------
N_TREND_RUNS = 7

prior_run_rows = (
    spark.table(summary_table)
    .filter(col("run_date") < lit(latest_run_date_val))
    .select("run_id", "run_date", "run_timestamp")
    .distinct()
    .orderBy(col("run_timestamp").desc())
    .select("run_id", "run_date")
    .limit(N_TREND_RUNS)
    .collect()
)

trend_dates = []  # date used for each trend column (or None if no run available)

for i in range(N_TREND_RUNS):
    trend_col = f"trend_{i}"

    if i < len(prior_run_rows):
        prior_run_id   = prior_run_rows[i]["run_id"]
        prior_run_date = str(prior_run_rows[i]["run_date"])
        trend_dates.append(prior_run_date)

        prior_df = (
            spark.table(summary_table)
            .filter(col("run_id") == prior_run_id)
            .select("field", col("similarity_pct").alias(f"prior_similarity_pct_{i}"))
            .toPandas()
        )
        df = df.merge(prior_df, on="field", how="left")
        df[trend_col] = df["similarity_pct"] - df[f"prior_similarity_pct_{i}"]
    else:
        trend_dates.append(None)
        df[trend_col] = pd.NA

# friendly column labels, e.g. "Trend (2026-08-12)"
trend_labels = {
    f"trend_{i}": (f"Trend ({trend_dates[i]})" if trend_dates[i] else f"Trend (run {i+1} n/a)")
    for i in range(N_TREND_RUNS)
}
trend_col_names = list(trend_labels.values())

field_cat_df = (
    spark.table(summary_table)
    .filter(col("run_id") == latest_run_id)
    .select("field", "field_category")
    .toPandas()
)

if "field_category" in df.columns:
    df = df.drop(columns=["field_category"])

df = field_cat_df.merge(df, on="field", how="right").sort_values(by=["field_category","field"])

cols = ["field_category"] + [c for c in df.columns if c != "field_category"]
df = df[cols]

report = df.rename(columns={
    "field_category":  "Field Category",
    "field":           "Field Name",
    "MATCH":           "# Match",
    "MISMATCH":        "# Mismatch",
    "MISSING":         "# Missing",
    "EXTRA":           "# Extra",
    "total":           "Total",
    "similarity_pct":  "% Similarity",
    **trend_labels,
})[[
    "Field Category",
    "Field Name",
    "# Match",
    "# Mismatch",
    "# Missing",
    "# Extra",
    "Total",
    "% Similarity",
    *trend_col_names,
]]

def similarity_color(val):
    if   val >= 99: return "background-color: #c6efce; color: #000; font-weight: bold"
    elif val >= 95: return "background-color: #ffeb9c; color: #000; font-weight: bold"
    else:           return "background-color: #ffc7ce; color: #000; font-weight: bold"

def trend_color(val):
    if val is None or pd.isna(val):
        return ""
    if val > 0:
        return "background-color: #c6efce; color: #006100; font-weight: bold"
    elif val < 0:
        return "background-color: #ffc7ce; color: #9c0006; font-weight: bold"
    return ""

try:
    styled = (
        report.style
        .map(similarity_color, subset=["% Similarity"])
        .map(trend_color, subset=trend_col_names)
    )
except AttributeError:
    styled = (
        report.style
        .applymap(similarity_color, subset=["% Similarity"])
        .applymap(trend_color, subset=trend_col_names)
    )

fmt_dict = {
    "# Match":       "{:,}",
    "# Mismatch":    "{:,}",
    "# Missing":     "{:,}",
    "# Extra":       "{:,}",
    "Total":         "{:,}",
    "% Similarity":  "{:.2f}%",
    **{name: "{:+.2f}%" for name in trend_col_names},
}

styled = (
    styled
    .format(fmt_dict, na_rep="—")
    .hide(axis="index")
    .set_table_styles([
        {"selector": "table",
         "props": [("border-collapse", "collapse"),
                   ("font-family", "Arial, sans-serif"),
                   ("font-size", "13px"),
                   ("width", "100%")]},
        {"selector": "th",
         "props": [("background-color", "#2e4057"),
                   ("color", "white"),
                   ("font-weight", "bold"),
                   ("padding", "8px 14px"),
                   ("text-align", "left"),
                   ("border", "1px solid #ccc")]},
        {"selector": "td",
         "props": [("padding", "7px 14px"),
                   ("border", "1px solid #ddd")]},
    ])
)

#summary_cards = f"""
#<div style="display:flex;gap:16px;margin-bottom:20px;font-family:Arial,sans-serif">
#  <div style="background:#f0f4f8;border-radius:8px;padding:14px 24px;text-align:center">
#    <div style="font-size:22px;font-weight:bold;color:#2e4057">{match_total:,}</div>
#    <div style="font-size:11px;color:#888;margin-top:2px">MATCHED</div>
#  </div>
#  <div style="background:#fff3cd;border-radius:8px;padding:14px 24px;text-align:center">
#    <div style="font-size:22px;font-weight:bold;color:#856404">{mismatch_total:,}</div>
#    <div style="font-size:11px;color:#888;margin-top:2px">MISMATCHED</div>
#  </div>
#  <div style="background:#f8d7da;border-radius:8px;padding:14px 24px;text-align:center">
##    <div style="font-size:22px;font-weight:bold;color:#842029">{extra_total:,}</div>
#   <div style="font-size:11px;color:#888;margin-top:2px">EXTRA IN ANCHOR</div>
#  </div>
#  <div style="background:#d1ecf1;border-radius:8px;padding:14px 24px;text-align:center">
#    <div style="font-size:22px;font-weight:bold;color:#0c5460">{missing_total:,}</div>
#    <div style="font-size:11px;color:#888;margin-top:2px">MISSING IN STREAM</div>
#  </div>
#</div>

# ── Dataset-wide value-level similarity ────────────────────────────────────
# The row-level cards above count PARTNUMBERS (whole rows). These cards count
# VALUES -- summing MATCH/MISMATCH/MISSING/EXTRA across every field of every
# matched row, so a row with 269/270 fields matching contributes 269 to
# MATCH and 1 to whichever bucket the drifted field falls in, instead of
# being lumped into a single row-level MISMATCH. Ratios sum to 100% by
# construction (Untitled-2.py's _valuestat_core guarantees m+mm+miss+ext=total).
# `df` here is still the per-field summary rows -- we sum across the fields
# in pandas rather than issuing another Spark query.
total_match_values    = int(df["MATCH"].sum())
total_mismatch_values = int(df["MISMATCH"].sum())
total_missing_values  = int(df["MISSING"].sum())
total_extra_values    = int(df["EXTRA"].sum())
total_values          = int(df["total"].sum())

def _value_pct(v):
    return (100.0 * v / total_values) if total_values else 0.0

overall_match_pct    = _value_pct(total_match_values)
overall_mismatch_pct = _value_pct(total_mismatch_values)
overall_missing_pct  = _value_pct(total_missing_values)
overall_extra_pct    = _value_pct(total_extra_values)

value_level_cards = f"""
<div style="font-family:Arial,sans-serif;font-size:11px;color:#888;margin:6px 0 6px 2px;letter-spacing:0.5px">
  DATASET-WIDE VALUE-LEVEL SIMILARITY &nbsp;<span style="color:#bbb">({total_values:,} values compared)</span>
</div>
<div style="display:flex;gap:16px;margin-bottom:22px;font-family:Arial,sans-serif">
  <div style="background:#e8f5e9;border-radius:8px;padding:14px 24px;text-align:center;min-width:150px">
    <div style="font-size:22px;font-weight:bold;color:#1b5e20">{overall_match_pct:.2f}%</div>
    <div style="font-size:11px;color:#888;margin-top:2px">MATCH</div>
    <div style="font-size:10px;color:#aaa;margin-top:1px">{total_match_values:,}</div>
  </div>
  <div style="background:#fff8e1;border-radius:8px;padding:14px 24px;text-align:center;min-width:150px">
    <div style="font-size:22px;font-weight:bold;color:#e65100">{overall_mismatch_pct:.2f}%</div>
    <div style="font-size:11px;color:#888;margin-top:2px">MISMATCH</div>
    <div style="font-size:10px;color:#aaa;margin-top:1px">{total_mismatch_values:,}</div>
  </div>
  <div style="background:#e1f5fe;border-radius:8px;padding:14px 24px;text-align:center;min-width:150px">
    <div style="font-size:22px;font-weight:bold;color:#01579b">{overall_missing_pct:.2f}%</div>
    <div style="font-size:11px;color:#888;margin-top:2px">MISSING</div>
    <div style="font-size:10px;color:#aaa;margin-top:1px">{total_missing_values:,}</div>
  </div>
  <div style="background:#fce4ec;border-radius:8px;padding:14px 24px;text-align:center;min-width:150px">
    <div style="font-size:22px;font-weight:bold;color:#880e4f">{overall_extra_pct:.2f}%</div>
    <div style="font-size:11px;color:#888;margin-top:2px">EXTRA</div>
    <div style="font-size:10px;color:#aaa;margin-top:1px">{total_extra_values:,}</div>
  </div>
</div>"""

_dates_present = [d for d in trend_dates if d]
if _dates_present:
    trend_note = f"&nbsp;|&nbsp; Trend vs last {len(_dates_present)} run(s): <strong>{', '.join(_dates_present)}</strong>"
else:
    trend_note = "&nbsp;|&nbsp; Trend: <strong>no prior runs to compare</strong>"

html_report = f"""<!DOCTYPE html>
<html>
<body style="font-family:Arial,sans-serif;padding:0;margin:0;background:#fff">
  <div style="padding:24px 28px">
    <h2 style="color:#2e4057;margin:0 0 4px 0;font-size:20px">Field Level Comparison Report - Style</h2>
    <p style="color:#888;font-size:12px;margin:0 0 20px 0">
      Run: <strong>{latest_run_id}</strong> &nbsp;|&nbsp; Date: <strong>{latest_run_date_timestamp}</strong>{trend_note}
    </p>
    {value_level_cards}
    {styled.to_html()}
  </div>
</body>
</html>"""


In [0]:
displayHTML(html_report)

Field Category,Field Name,# Match,# Mismatch,# Missing,# Extra,Total,% Similarity,Trend (2026-09-04),Trend (2026-09-06),Trend (2026-09-05),Trend (2026-09-03),Trend (2026-09-02),Trend (2026-09-01),Trend (2026-08-31)
Attribution,attributes,"4,515,996","26,320","52,369","745,528","5,340,213",84.70%,-0.21%,—,—,—,—,—,—
Attribution,customSkuAttributes,"2,797,319","22,863","46,380","735,529","3,602,091",77.67%,-0.17%,—,—,—,—,—,—
Attribution,defAttributes,0,0,0,0,0,100.00%,+0.00%,—,—,—,—,—,—
Attribution,floatFacets,"99,006",0,"3,247","3,250","105,503",97.80%,-1.42%,—,—,—,—,—,—
Attribution,numberFacets,0,0,0,0,0,100.00%,+0.00%,—,—,—,—,—,—
Attribution,searchAttributes,"3,149,750",0,"20,666","42,839","3,213,255",98.15%,-0.01%,—,—,—,—,—,—
Attribution,stringFacets,"1,693,682",0,"6,381","10,417","1,710,480",99.01%,-0.02%,—,—,—,—,—,—
Caddyshack,assetSeoUrl,"163,342",363,4,39,"163,748",99.75%,-0.06%,—,—,—,—,—,—
Caddyshack,catgroupSeq,"5,977","2,666","1,353,828","4,478,921","5,841,392",0.13%,+0.00%,—,—,—,—,—,—
Caddyshack,dsgCatgroups,"1,701,367",0,"134,009","45,858","1,881,234",90.92%,-1.30%,—,—,—,—,—,—


In [0]:
# Email delivery — uncomment when Power Automate URL is confirmed
# payload = {
#     "recipients": ", ".join(RECIPIENTS),
#     "subject":    f"Field Level Comparison Report — {latest_run_date}",
#     "body":       html_report,
# }
# response = requests.post(FLOW_URL, json=payload)
# if response.status_code in (200, 202):
#     print(f"Report sent to: {', '.join(RECIPIENTS)}")
# else:
#     raise Exception(f"Power Automate returned {response.status_code}: {response.text}")
